<a href="https://colab.research.google.com/github/alexcho2823-del/ds2002-fa26/blob/main/notebooks/04-json-apis/lab07_public_api_to_clean_table.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [1]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }
    last_problem = None
    for attempt in range(tries):
        try:
            r = requests.get(BASE, params=params, timeout=10)
        except (requests.ConnectionError, requests.Timeout) as e:
            last_problem = f'network error: {e}'
        else:
            if r.status_code == 200:
                return pd.DataFrame(r.json()['daily'])
            if r.status_code not in RETRY_STATUSES:
                # 400/404 etc. = our bug, fail immediately
                raise requests.HTTPError(f'{r.status_code} from API: {r.text[:200]}')
            last_problem = f'HTTP {r.status_code}'
        if attempt < tries - 1:
            time.sleep(2 ** attempt)   # 1s, 2s, 4s...
    raise RuntimeError(f'get_weather failed after {tries} tries ({last_problem})')



this function retries on the rate limits, network drops, and server errors with a delay that is growing constantly, but raises right away on the errors like 400 and 404 that mean the request is wrong.

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [2]:
START, END = '2026-09-05', '2026-10-03'
DATES = [START, END]
used_fallback = False
pulled_at = pd.Timestamp.now(tz='America/New_York')

try:
    raw = get_weather(START, END)
    wx = raw[raw['time'].isin(DATES)].reset_index(drop=True)
except RuntimeError as e:
    print(f'API call failed, will use fallback: {e}')
    wx = None


one request would cover both of the dates, and only a genuine failure (retries used up) would trigger the fall back. Therefore a bug in my request would stil crash loud.

### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [3]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-03'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})

if wx is None:
    used_fallback = True
    wx = FALLBACK.copy()

wx['source'] = 'fallback sample' if used_fallback else 'open-meteo'
wx

,time,temperature_2m_max,precipitation_sum,source
0,2026-09-05,29.4,2.5,open-meteo
1,2026-10-03,22.0,14.7,open-meteo


the source column will let the user know if the what they are reading in the row is real data or the sample data.

### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [4]:
wx = (
    wx.rename(columns={
        'time': 'date',
        'temperature_2m_max': 'temp_max',
        'precipitation_sum': 'precip_mm',
    })
    .assign(date=lambda d: pd.to_datetime(d['date']))
    [['date', 'temp_max', 'precip_mm', 'source']]
)
wx

,date,temp_max,precip_mm,source
0,2026-09-05,29.4,2.5,open-meteo
1,2026-10-03,22.0,14.7,open-meteo


This simply creates my own column names for better readability and organization.

### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [5]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')

weather table is safe to join.


This is an early fail safe, if anything goes wrong it will fail here and not something down the road as problems can quickly grow bigger

### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [6]:
wet = wx.loc[wx['precip_mm'].idxmax()]
dry = wx.loc[wx['precip_mm'].idxmin()]
diff = wet['precip_mm'] - dry['precip_mm']

if diff == 0:
    msg = f"Both days had the same rainfall ({wet['precip_mm']:.1f} mm)."
else:
    msg = (f"{wet['date']:%A, %B %d, %Y} was the wetter day, with {wet['precip_mm']:.1f} mm of rain, "
           f"{diff:.1f} mm more than {dry['date']:%A, %B %d, %Y} ({dry['precip_mm']:.1f} mm).")
if used_fallback:
    msg += " (Note: based on SAMPLE data, not real observations.)"
print(msg)

Saturday, October 03, 2026 was the wetter day, with 14.7 mm of rain, 12.2 mm more than Saturday, September 05, 2026 (2.5 mm).


THis prints a readable sentence following the prompt

### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [7]:
print('Endpoint:    ', BASE)
print('Coordinates:  38.03, -78.51 (Charlottesville, VA)')
print(f'Date range:   {START} to {END} (kept {", ".join(DATES)})')
print('Pulled at:   ', pulled_at.strftime('%Y-%m-%d %H:%M %Z'))
print('Data status: ', 'FALLBACK SAMPLE (not real)' if used_fallback else 'Real data from Open-Meteo archive')

Endpoint:     https://archive-api.open-meteo.com/v1/archive
Coordinates:  38.03, -78.51 (Charlottesville, VA)
Date range:   2026-09-05 to 2026-10-03 (kept 2026-09-05, 2026-10-03)
Pulled at:    2026-10-13 19:04 EDT
Data status:  Real data from Open-Meteo archive


THis is just the source and tells any future user or scholar where my data came from and whatnot.

### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

If the get_weather returned something empty like empty DataFrame instead of raising, the failure passes silently through the rest of the pipeline. Merging the empty frame onto 200 days of sales with a left join will keep all of the 200 sales rows, but every rain value would end up being NaN; with an inner join, the result would have zero rows. Regardless of the way, the plotting step wouldn’t complain. Most libraries would usually quietly drop NaN values, so the rain-vs-revenue chart would come out blank. If someone had added a fillna(0) to clean up the missing values, every day would show 0 mm of rain, with revenue varying normally, which looks like a clear finding that rain has no effect on sales.

This won't be noticed by anyone. Because there are no error messages or traces shown to the user, and because the notebook will run through each cell from top to bottom, the bar graph will render appropriately with the correct x & y axis titles, and "no relationship" is a reasonable outcome that people would probably not ever suspect anything was amiss. This is also why the function needs to fail loudly if something goes wrong, and why the validation checks in Part 5 of the lab manual are important. An immediate loud crash gets your attention and makes you address the issue at hand now, while an empty table that "works" turns a failure of collecting data into a false finding.

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [8]:
CITIES = {'Charlottesville': (38.03, -78.51), 'Orlando': (28.54, -81.38)}
frames = []

for city, (lat, lon) in CITIES.items():
    raw_c = get_weather(START, END, lat=lat, lon=lon)
    df = (raw_c[raw_c['time'].isin(DATES)]
          .rename(columns={'time': 'date', 'temperature_2m_max': 'temp_max',
                           'precipitation_sum': 'precip_mm'}))
    df['date'] = pd.to_datetime(df['date'])
    df['city'] = city
    df['source'] = 'open-meteo'
    frames.append(df)
    time.sleep(1)  # be polite to the free API

wx_long = pd.concat(frames, ignore_index=True)[['city', 'date', 'temp_max', 'precip_mm', 'source']]
assert len(wx_long) == 4
wx_long

,city,date,temp_max,precip_mm,source
0,Charlottesville,2026-09-05,29.4,2.5,open-meteo
1,Charlottesville,2026-10-03,22.0,14.7,open-meteo
2,Orlando,2026-09-05,32.4,3.9,open-meteo
3,Orlando,2026-10-03,30.4,13.4,open-meteo


This builds the long-format, one-row-per-city-per-date table the midterm uses to compare stores in different places.